In [1]:
import numpy as np 
import pandas as pd 
import re
import nltk
from nltk.corpus import stopwords

In [4]:
def import_y_filter(location: str, vet_business_ids: set | None = None) -> pd.DataFrame:
    
    if location.endswith('.json'):
        full_df = pd.read_json(location, lines = True)  
    elif location.endswith('.csv'):
        full_df = pd.read_csv(location)
    else:
        raise ValueError('Unusable file type')  

    
    if any(keyword in location for keyword in ('checkin', 'tip', 'review')):
        if vet_business_ids is None:
            raise ValueError('vet_business_ids required to filter this file type')
        filtered_df = full_df[full_df['business_id'].isin(vet_business_ids)].copy()
    
    
    else:
        filtered_df = full_df[full_df['categories'].str.contains('Veterinarian', na = False)].copy()

    return filtered_df

def import_review_chunked(location, vet_ids) -> pd.DataFrame:
    chunks = []
    for chunk in pd.read_json(location, lines=True, chunksize = 100000):
        filtered = chunk[chunk['business_id'].isin(vet_ids)]
        chunks.append(filtered)
    return pd.concat(chunks, ignore_index=True)

def import_user_chunked(location, vet_user_ids) -> pd.DataFrame:
    chunks = []
    for chunk in pd.read_json(location, lines=True, chunksize = 100000):
        filtered = chunk[chunk['user_id'].isin(vet_user_ids)]
        chunks.append(filtered)
    return pd.concat(chunks, ignore_index=True)

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)  
    text = re.sub(r'\d+', '', text)       
    return text

def remove_stopwords(text):
    return ' '.join([word for word in text.split() if word not in stop_words])


In [5]:
business = import_y_filter(
    '/kaggle/input/datasets/organizations/yelp-dataset/yelp-dataset/yelp_academic_dataset_business.json'
)

vet_ids = set(business['business_id'])


check_in = import_y_filter(
    '/kaggle/input/datasets/organizations/yelp-dataset/yelp-dataset/yelp_academic_dataset_checkin.json',
    vet_business_ids = vet_ids
)

tip = import_y_filter(
    '/kaggle/input/datasets/organizations/yelp-dataset/yelp-dataset/yelp_academic_dataset_tip.json',
    vet_business_ids = vet_ids
)

In [6]:
reviews = import_review_chunked(
    '/kaggle/input/datasets/organizations/yelp-dataset/yelp-dataset/yelp_academic_dataset_review.json',
    vet_ids = business['business_id'])

In [7]:
vet_user_ids = set(reviews['user_id'])

user = import_user_chunked(
    '/kaggle/input/datasets/organizations/yelp-dataset/yelp-dataset/yelp_academic_dataset_user.json',
    vet_user_ids = vet_user_ids
)

In [9]:
reviews['clean_text'] = reviews['text'].apply(clean_text)

In [10]:
# nltk.download('stopwords')
# stop_words = set(stopwords.words('english'))
# pd.DataFrame(stop_words, columns=['word']).to_csv('stop_words.csv', index=False)

stop_words = set(pd.read_csv('/kaggle/input/datasets/micahluftig/stop-words/stop_words.csv')['word'])

specific_words = ['vet', 'dog', 'dr', 'cat', 'us', 'would', 'pet', 'dogs', 'get',
                   'always', 'back', 'one', 'took', 'like', 'take', 'pets', 'even', 'go', 'staff', 'animal', 'really',
                   'vets', 'also', 'animals', 'cats', 'caring', 'got', 'going', 'day',
                   'dont', 'didnt', 'doesnt', 'wont', 'wasnt', 'isnt', 'couldnt', 'wouldnt'
                 ]

stop_words.update(specific_words) 


reviews['clean_text'] = reviews['clean_text'].apply(remove_stopwords)

In [2]:
categories = {
    'wait_time':      r'\b(?:wait|waiting|slow|long|hours|delay)\b',
    'compassion':     r'\b(?:compassion|caring|kind|empathy|gentle|comfort)\b',
    'communication': r'\b(?:explain|told|informed|discuss|understand\w*|understood)\b',
    'cost': r'\b(?:expensive|price|cost|bill|charge|affordable|estimate|overestimate)\b',
    'end_of_life':    r'\b(?:euthanasia|passing|put down|goodbye|cremation)\b',
    'diagnosis':      r'\b(?:diagnos|misdiagnos|treatment|prognosis)\b'
}

for category, pattern in categories.items():
    reviews[category] = reviews['clean_text'].str.contains(pattern, regex=True).astype(int)

NameError: name 'reviews' is not defined

In [13]:
business.to_csv('yelp_veterinary_business.csv', index=False)
check_in.to_csv('yelp_veterinary_check_in.csv', index=False)
tip.to_csv('yelp_veterinary_tip.csv', index=False)
reviews.to_csv('yelp_veterinary_reviews.csv', index=False)
user.to_csv('yelp_veterinary_user.csv', index=False)